In [1]:
from pathlib import Path
import os, stat, glob, traceback
import numpy as np
import pandas as pd
import arviz as az
from cmdstanpy import CmdStanModel
import matplotlib.pyplot as plt

/Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
os.environ.pop('CXX', None)
os.environ.pop('CC', None)

In [3]:
# Paths and helper to compile models
stan_dir = Path('../stan').resolve()
out_root = Path.cwd() / 'stan_output'
out_root.mkdir(parents=True, exist_ok=True)
models = {
    'minimal': {'file': stan_dir / 'minimal.stan', 'outdir': out_root / 'minimal'},
    'province': {'file': stan_dir / 'logistic_province.stan', 'outdir': out_root / 'province'},
    'full': {'file': stan_dir / 'logistic_full.stan', 'outdir': out_root / 'full'},
    'response': {'file': stan_dir / 'logistic_response.stan', 'outdir': out_root / 'response'},
}
for m in models.values():
    m['outdir'].mkdir(parents=True, exist_ok=True)

def compile_model(stan_path):
    print('Compiling', stan_path.name)
    mdl = CmdStanModel(stan_file=str(stan_path))
    mode = os.stat(mdl.exe_file).st_mode
    os.chmod(mdl.exe_file, mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return mdl

### Data Prep

In [4]:
# Load data and prepare stan_data objects (one-off)
df = pd.read_csv('../data/fire_db.csv')
# minimal
df0 = df[['any_evacuation', 'log_fire_size_ha', 'fn_indicator']].dropna().copy()
stan_data_0 = {
    'N': len(df0),
    'y': df0['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df0['log_fire_size_ha'].astype(float).tolist(),
    'fn_indicator': df0['fn_indicator'].astype(int).tolist(),
}
# province
df_proc = df.dropna(subset=['any_evacuation','log_fire_size_ha','fn_indicator','province']).copy()
prov_cat = pd.Categorical(df_proc['province'])
df_proc['prov_idx'] = prov_cat.codes + 1
stan_data_province = {
    'N': len(df_proc),
    'P': len(prov_cat.categories),
    'province': df_proc['prov_idx'].astype(int).tolist(),
    'y': df_proc['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_proc['log_fire_size_ha'].astype(float).tolist(),
    'fn_indicator': df_proc['fn_indicator'].astype(int).tolist(),
}
# full
df_full = df.dropna(subset=['any_evacuation','log_fire_size_ha','fn_indicator','dist_to_fn_km','n_fn_20km','fire_cause','fire_type','protection_zone','province']).copy()
prov_cat_full = pd.Categorical(df_full['province'])
cause_cat = pd.Categorical(df_full['fire_cause'])
type_cat = pd.Categorical(df_full['fire_type'])
prot_cat = pd.Categorical(df_full['protection_zone'])

_log_fire = df_full['log_fire_size_ha'].astype(float).to_numpy()
_dist = df_full['dist_to_fn_km'].astype(float).to_numpy()
# _nfn = df_full['n_fn_20km'].astype(float).to_numpy()
stan_data_full = {
    'N': len(df_full),
    'P': len(prov_cat_full.categories),
    'province': (prov_cat_full.codes + 1).tolist(),
    'y': df_full['any_evacuation'].astype(int).tolist(),
    'log_fire_size': _log_fire.tolist(),
    'dist_to_fn_km': _dist.tolist(),
    # 'n_fn_20km': _nfn.tolist(),
    'fn_indicator': df_full['fn_indicator'].astype(int).tolist(),
    # 'K_cause': len(cause_cat.categories),
    # 'fire_cause': (cause_cat.codes + 1).tolist(),
    # 'K_type': len(type_cat.categories),
    # 'fire_type': (type_cat.codes + 1).tolist(),
    'K_prot': len(prot_cat.categories),
    'protection_zone': (prot_cat.codes + 1).tolist(),
}
# response
df_resp = df.dropna(subset=['any_evacuation','log_fire_size_ha','fn_indicator','dist_to_fn_km','n_fn_20km','fire_cause','fire_type','protection_zone','response_type','province']).copy()
prov_cat_r = pd.Categorical(df_resp['province'])
cause_cat_r = pd.Categorical(df_resp['fire_cause'])
type_cat_r = pd.Categorical(df_resp['fire_type'])
resp_cat = pd.Categorical(df_resp['response_type'])
prot_cat_r = pd.Categorical(df_resp['protection_zone'])
stan_data_response = {
    'N': len(df_resp),
    'P': len(prov_cat_r.categories),
    'province': (prov_cat_r.codes + 1).tolist(),
    'y': df_resp['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_resp['log_fire_size_ha'].astype(float).tolist(),
    'dist_to_fn_km': df_resp['dist_to_fn_km'].astype(float).tolist(),
    # 'n_fn_20km': df_resp['n_fn_20km'].astype(int).tolist(),
    'fn_indicator': df_resp['fn_indicator'].astype(int).tolist(),
    # 'K_cause': len(cause_cat_r.categories),
    # 'fire_cause': (cause_cat_r.codes + 1).tolist(),
    # 'K_type': len(type_cat_r.categories),
    # 'fire_type': (type_cat_r.codes + 1).tolist(),
    'K_response': len(resp_cat.categories),
    'response_type': (resp_cat.codes + 1).tolist(),
    'K_prot': len(prot_cat_r.categories),
    'protection_zone': (prot_cat_r.codes + 1).tolist(),
}
print('Prepared data sizes: minimal', stan_data_0['N'], 'province', stan_data_province['N'], 'full', stan_data_full['N'], 'response', stan_data_response['N'])

Prepared data sizes: minimal 15203 province 15203 full 15203 response 15203


In [5]:
# Helper to run, convert and save InferenceData for a single model
def run_and_save(name, mdl, data, outdir, chains=4, warmup=500, sampling=500, adapt_delta=0.95, show_console=False):
    print('Running', name)
    fit = mdl.sample(
        data=data,
        chains=chains,
        parallel_chains=min(chains, os.cpu_count() or 1),
        iter_warmup=warmup,
        iter_sampling=sampling,
        seed=20260417,
        show_console=show_console,
        adapt_delta=adapt_delta,
        output_dir=str(outdir),
        save_warmup=False,
    )
    idata = az.from_cmdstanpy(posterior=fit, posterior_predictive='y_rep', log_likelihood='log_lik')
    fn = outdir / 'idata.nc'
    idata.to_netcdf(str(fn))
    print('Saved idata to', fn)
    return fit, idata

## Fit Models

In [6]:
chains = 4
warmup = 500
sampling = 1000
adapt_delta = 0.95

In [7]:
mdl_min = compile_model(models['minimal']['file'])
fit_min, idata_min = run_and_save('minimal', mdl_min, stan_data_0, models['minimal']['outdir'], chains=chains, warmup=warmup, sampling=sampling, adapt_delta=adapt_delta)

Compiling minimal.stan
Running minimal


19:41:25 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]





chain 1:   7%|▋         | 100/1500 [00:03<00:44, 31.25it/s, (Warmup)]


chain 1:  13%|█▎        | 200/1500 [00:05<00:35, 37.03it/s, (Warmup)]


chain 1:  20%|██        | 300/1500 [00:07<00:25, 46.95it/s, (Warmup)]


chain 1:  27%|██▋       | 400/1500 [00:08<00:20, 52.50it/s, (Warmup)]





chain 1:  33%|███▎      | 500/1500 [00:10<00:20, 48.17it/s, (Sampling)]


chain 1:  40%|████      | 600/1500 [00:13<00:20, 44.20it/s, (Sampling)]


chain 1:  47%|████▋     | 700/1500 [00:16<00:18, 42.26it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [00:18<00:17, 39.88it/s, (Sampling)]


chain 1:  60%|██████    | 900/1500 [00:21<00:15, 38.93it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [00:25<00:14, 34.45it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [00:28<00:12, 33.01it/s, (Sampling)]


chain 1:  80%|████████  | 1200/1500 [00:31<00:08, 33.56it/s, (Samplin


19:42:06 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/minimal/idata.nc


In [8]:
mdl_prov = compile_model(models['province']['file'])
fit_prov, idata_prov = run_and_save('province', mdl_prov, stan_data_province, models['province']['outdir'], chains=chains, warmup=warmup, sampling=sampling, adapt_delta=adapt_delta)

Compiling logistic_province.stan
Running province


19:42:28 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]


chain 1:   7%|▋         | 100/1500 [00:16<03:49,  6.10it/s, (Warmup)]


chain 1:  13%|█▎        | 200/1500 [00:30<03:14,  6.70it/s, (Warmup)]


chain 1:  20%|██        | 300/1500 [00:40<02:33,  7.82it/s, (Warmup)]


chain 1:  27%|██▋       | 400/1500 [00:51<02:12,  8.30it/s, (Warmup)]


chain 1:  33%|███▎      | 500/1500 [01:03<02:01,  8.22it/s, (Sampling)]







chain 1:  47%|████▋     | 700/1500 [01:22<01:24,  9.51it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [01:32<01:13,  9.51it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [01:58<00:58,  8.49it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [02:10<00:48,  8.31it/s, (Sampling)]


chain 1:  87%|████████▋ | 1300/1500 [02:32<00:22,  8.71it/s, (Sampling)]


chain 1: 100%|██████████| 1500/1500 [02:47<00:00, 10.62it/s, (Sampling)]




chain 1: 100%|██████████| 1500/1500 [02:57<00:00, 10.62it/s, (Samp

19:45:49 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/province/idata.nc


In [9]:
mdl_full = compile_model(models['full']['file'])
fit_full, idata_full = run_and_save('full', mdl_full, stan_data_full, models['full']['outdir'], chains=chains, warmup=warmup, sampling=sampling, adapt_delta=adapt_delta)

Compiling logistic_full.stan
Running full


19:46:12 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]


chain 1:   7%|▋         | 100/1500 [04:28<1:02:44,  2.69s/it, (Warmup)]



chain 1:  20%|██        | 300/1500 [06:14<19:46,  1.01it/s, (Warmup)]

chain 1:  27%|██▋       | 400/1500 [06:42<12:59,  1.41it/s, (Warmup)]


chain 1:  33%|███▎      | 501/1500 [07:16<09:33,  1.74it/s, (Sampling)]


chain 1:  40%|████      | 600/1500 [07:44<06:45,  2.22it/s, (Sampling)]


chain 1:  47%|████▋     | 700/1500 [08:17<05:24,  2.47it/s, (Sampling)]






chain 1:  53%|█████▎    | 800/1500 [08:47<04:17,  2.72it/s, (Sampling)]


chain 1:  60%|██████    | 900/1500 [09:22<03:38,  2.75it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [10:19<02:07,  3.13it/s, (Sampling)]

chain 1:  80%|████████  | 1200/1500 [10:45<01:30,  3.33it/s, (Sampling)]


chain 1:  87%|████████▋ | 1300/1500 [11:12<00:58,  3.43it/s, (Sampling)]


chain 1:  93%|█████████▎| 1400/1500 [11:40<00:28,  3.47it/s, (Sampl


19:59:25 - cmdstanpy - INFO - CmdStan done processing.



Saved idata to /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/full/idata.nc
